# Overview

**Paper**: *Chain Drugstore Sales Prediction Method Based on the Fusion of Self-Attention Mechanism and LightGBM*

**Authors**: Zhiyong Zeng, Weijie Yang, Min Wang, Mengling Zhu, Tao Feng (2025)

**Objective**: Reproduce the experimental results from Section 4.3.1 (Rossmann Dataset) of the Zeng et al. (2025) paper, specifically comparing XGBoost, LightGBM, TS-XGBoost, and TS-LGBM models using the RMSPE metric.

**Principle**: Pure reproduction without assumptions or optimization. Models are implemented precisely as described in the paper's methodology section using the Rossmann dataset.

**Paper Reference Values (Table 7 - Rossmann Dataset)**:

| Model | RMSPE |
| :--- | :--- |
| XGBoost | 0.274 |
| LightGBM | 0.273 |
| TS-XGBoost | 0.253 |
| TS-LGBM | 0.248 |


# Model Architectures


# Reference-Based Exploration

Exploration follows the methodology from Zeng et al. (2025). Preprocessing involves cleaning the Rossmann dataset (removing zero-sales records), applying a chronological split (900 days train / 42 days test), engineering features, and defining the RMSPE metric. Evaluated models include XGBoost (baseline), LightGBM (baseline), TS-XGBoost, and TS-LGBM.


## Preprocessing

Data loading and processing strictly follow the paper methodology: removing `open==0` or `sales==0` records, temporal data partitioning (900 training days / 42 testing days), and defining the RMSPE evaluation metric.


In [1]:
import sys
import os
import warnings
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if PROJECT_ROOT.name == 'explore' or PROJECT_ROOT.name == 'notebooks' or PROJECT_ROOT.name == 'Notebooks-to-transfers':
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT.name == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import math

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")
print("All imports successful.")

Project root: D:\Work-Env\ITEC\forecasting-medicine-public
PyTorch version: 2.13.0+cpu
All imports successful.


In [2]:
# Load Rossmann datasets
train_path = os.path.join(PROJECT_ROOT, "data", "raw", "rossmann", "train.csv")
store_path = os.path.join(PROJECT_ROOT, "data", "raw", "rossmann", "store.csv")

print("Loading Rossmann datasets...")
df_train = pd.read_csv(train_path, parse_dates=['Date'])
df_store = pd.read_csv(store_path)

# Merge train and store data
df_merged = pd.merge(df_train, df_store, on='Store', how='left')

n_dates = df_merged["Date"].nunique()
complete_stores = (
    df_merged.groupby("Store")
    .size()
    .loc[lambda counts: counts == n_dates]
    .index
)

df_cleaned = df_merged[df_merged["Store"].isin(complete_stores)].copy()
print(f"Dataset shape after cleaning: {df_cleaned.shape}")
print(f"Number of unique stores: {df_cleaned['Store'].nunique()}")
print(f"Date range: {df_cleaned['Date'].min()} to {df_cleaned['Date'].max()}")

Loading Rossmann datasets...
Dataset shape after cleaning: (879828, 18)
Number of unique stores: 934
Date range: 2013-01-01 00:00:00 to 2015-07-31 00:00:00


In [3]:
# Encode categorical features
df_cleaned['StoreType'] = df_cleaned['StoreType'].map({'a': 0, 'b': 1, 'c': 2, 'd': 3}).astype(int)
df_cleaned['Assortment'] = df_cleaned['Assortment'].map({'a': 0, 'b': 1, 'c': 2}).astype(int)
df_cleaned['StateHoliday'] = df_cleaned['StateHoliday'].astype(str).replace('0', '0').map({'0': 0, 'a': 1, 'b': 2, 'c': 3}).fillna(0).astype(int)

# Extract temporal features
df_cleaned['Year'] = df_cleaned['Date'].dt.year
df_cleaned['Month'] = df_cleaned['Date'].dt.month
df_cleaned['Day'] = df_cleaned['Date'].dt.day
df_cleaned['DayOfWeek'] = df_cleaned['Date'].dt.dayofweek
df_cleaned['WeekOfYear'] = df_cleaned['Date'].dt.isocalendar().week

# Sort by store and date for temporal consistency
df_cleaned = df_cleaned.sort_values(['Store', 'Date']).reset_index(drop=True)

print("Feature engineering completed.")
print(df_cleaned[['Store', 'Date', 'Sales', 'DayOfWeek', 'Year', 'Month']].head(10))

Feature engineering completed.
   Store       Date  Sales  DayOfWeek  Year  Month
0      1 2013-01-01      0          1  2013      1
1      1 2013-01-02   5530          2  2013      1
2      1 2013-01-03   4327          3  2013      1
3      1 2013-01-04   4486          4  2013      1
4      1 2013-01-05   4997          5  2013      1
5      1 2013-01-06      0          6  2013      1
6      1 2013-01-07   7176          0  2013      1
7      1 2013-01-08   5580          1  2013      1
8      1 2013-01-09   5471          2  2013      1
9      1 2013-01-10   4892          3  2013      1


In [4]:
# Select features for modeling
feature_cols = ['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 
                 'Year', 'Month', 'Day', 'StoreType', 'Assortment', 
                 'CompetitionDistance', 'Promo2', 'WeekOfYear']

# Handle missing values in Competition columns
df_cleaned['CompetitionDistance'].fillna(df_cleaned['CompetitionDistance'].median(), inplace=True)
df_cleaned['CompetitionOpenSinceMonth'].fillna(0, inplace=True)
df_cleaned['CompetitionOpenSinceYear'].fillna(0, inplace=True)
df_cleaned['Promo2'].fillna(0, inplace=True)
df_cleaned['Promo2SinceWeek'].fillna(0, inplace=True)
df_cleaned['Promo2SinceYear'].fillna(0, inplace=True)

# Partition data: 900 days training, 42 days testing
# Get unique dates and sort them
unique_dates = sorted(df_cleaned['Date'].unique())
print(f"Total unique dates: {len(unique_dates)}")

split_date = unique_dates[899]  # Last date of first 900 days
test_start_date = unique_dates[900]  # First date of test set

print(f"Training period: up to {split_date}")
print(f"Test period: from {test_start_date} to {unique_dates[-1]}")

# Split data
df_train = df_cleaned[df_cleaned['Date'] <= split_date].copy()
df_test = df_cleaned[df_cleaned['Date'] > split_date].copy()

print(f"\nTraining set size: {df_train.shape[0]} samples")
print(f"Test set size: {df_test.shape[0]} samples")

Total unique dates: 942
Training period: up to 2015-06-19 00:00:00
Test period: from 2015-06-20 00:00:00 to 2015-07-31 00:00:00

Training set size: 840600 samples
Test set size: 39228 samples


In [5]:
def calculate_rmspe(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    
    mask = y_true > 0
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    
    percentage_errors = (y_true - y_pred) / y_true
    squared_errors = percentage_errors ** 2
    rmspe = np.sqrt(np.mean(squared_errors))
    
    return rmspe


def calculate_regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "RMSE": round(np.sqrt(mse), 2),
        "MSE": round(mse, 2),
        "RMSPE": round(calculate_rmspe(y_true, y_pred), 4),
        "R^2": round(r2_score(y_true, y_pred), 4),
        "MAE": round(mean_absolute_error(y_true, y_pred), 2),
    }

print("RMSPE metric function defined.")


RMSPE metric function defined.


## Modeling

Four distinct models are evaluated using the reference preprocessing: XGBoost baseline, LightGBM baseline, TS-XGBoost (featuring temporal parameter extraction), and TS-LGBM (the proposed architecture from the paper).


### XGBoost

According to paper Section 3.1.1, XGBoost operates as an ensemble learning algorithm utilizing gradient boosting optimization.


In [6]:
print("Training XGBoost model...")

# Prepare data for XGBoost
X_train = df_train[feature_cols].copy()
y_train = df_train['Sales'].copy()
X_test = df_test[feature_cols].copy()
y_test = df_test['Sales'].copy()

# Train XGBoost model
xgb_model = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train, verbose=0)

# Predictions
y_pred_xgb = xgb_model.predict(X_test)

# Calculate RMSPE
rmspe_xgb = calculate_rmspe(y_test, y_pred_xgb)

print(f"XGBoost RMSPE: {rmspe_xgb:.4f}")
print(f"Paper XGBoost RMSPE: 0.274")
print(f"Difference: {abs(rmspe_xgb - 0.274):.4f}")

Training XGBoost model...
XGBoost RMSPE: 0.3578
Paper XGBoost RMSPE: 0.274
Difference: 0.0838


### LightGBM

According to paper Section 3.1.2, LightGBM functions as a distributed gradient boosting framework built on CART decision trees.


In [7]:
print("Training LightGBM model...")

# Train LightGBM model
lgb_model = lgb.LGBMRegressor(
    max_depth=-1,
    learning_rate=0.1,
    n_estimators=100,
    boosting_type='gbdt',
    random_state=42,
    n_jobs=-1
)
lgb_model.fit(X_train, y_train)

# Predictions
y_pred_lgb = lgb_model.predict(X_test)

# Calculate RMSPE
rmspe_lgb = calculate_rmspe(y_test, y_pred_lgb)

print(f"LightGBM RMSPE: {rmspe_lgb:.4f}")
print(f"Paper LightGBM RMSPE: 0.273")
print(f"Difference: {abs(rmspe_lgb - 0.273):.4f}")

Training LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.026361 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 635
[LightGBM] [Info] Number of data points in the train set: 840600, number of used features: 13
[LightGBM] [Info] Start training from score 5817.628823
LightGBM RMSPE: 0.3489
Paper LightGBM RMSPE: 0.273
Difference: 0.0759


### TS-XGBoost

According to paper Section 3.1.3:
1. **Sliding Window Synthesis**: Transforms the training set into a synthetic dataset by interacting sales features with lagged historical data.
2. **Temporal Parameter Extraction**: Employs a Transformer encoder for multi-head attention processing.
3. **Training**: Integrates synthetic features into the gradient boosting process.


In [8]:
def create_sliding_window_features(df, feature_cols, window_size=7):
    """
    Create sliding window features for time-series data.
    
    Parameters:
    -----------
    df : DataFrame
        Input dataframe sorted by Store and Date
    feature_cols : list
        Feature columns to use
    window_size : int
        Size of sliding window (default: 7 days)
    
    Returns:
    --------
    DataFrame
        Dataframe with sliding window features
    """
    df_windowed = df.copy()
    
    # Group by store for windowing
    for store in df['Store'].unique():
        store_mask = df_windowed['Store'] == store
        store_indices = df_windowed[store_mask].index
        
        # Create lagged sales features
        for lag in range(1, window_size + 1):
            lagged_sales = df_windowed.loc[store_mask, 'Sales'].shift(lag)
            df_windowed.loc[store_mask, f'Sales_lag_{lag}'] = lagged_sales
    
    # Remove rows with NaN values from windowing
    df_windowed = df_windowed.dropna()
    
    return df_windowed

print("Creating sliding window features for training data...")
df_train_windowed = create_sliding_window_features(df_train, feature_cols, window_size=7)
print(f"Training set with window features: {df_train_windowed.shape}")

# For test set, we need to include actual previous values
df_combined = pd.concat([df_train, df_test], ignore_index=False).sort_values(['Store', 'Date']).reset_index(drop=True)
df_combined['original_index'] = range(len(df_combined))
df_combined_windowed = create_sliding_window_features(df_combined, feature_cols, window_size=7)

# Get indices of test set in combined dataset (before windowing)
test_indices_in_combined = df_combined[df_combined['Date'] > df_train['Date'].max()].index.tolist()
# Get indices in windowed dataset that correspond to test period
test_dates = df_test['Date'].unique()
df_test_windowed = df_combined_windowed[df_combined_windowed['Date'].isin(test_dates)].copy()

print(f"Test set with window features: {df_test_windowed.shape}")

Creating sliding window features for training data...
Training set with window features: (370595, 29)
Test set with window features: (17430, 30)


In [9]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism as described in Zeng et al. (2025).
    Used for temporal parameter extraction from time-series data.
    """
    def __init__(self, embed_dim, num_heads=8):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        self.head_dim = embed_dim // num_heads
        self.scale = 1 / math.sqrt(self.head_dim)
        
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        self.W_o = nn.Linear(embed_dim, embed_dim)
    
    def forward(self, Q, K, V):
        # Linear transformations
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)
        
        # Reshape for multi-head attention
        batch_size = Q.shape[0]
        Q = Q.reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.reshape(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Calculate attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn_weights = torch.softmax(scores, dim=-1)
        
        # Apply attention to values
        context = torch.matmul(attn_weights, V)
        context = context.transpose(1, 2).contiguous()
        context = context.reshape(batch_size, -1, self.embed_dim)
        
        # Output projection
        output = self.W_o(context)
        return output

print("Multi-Head Attention module defined.")

Multi-Head Attention module defined.


In [10]:
def extract_temporal_parameters(df_input, feature_cols, window_size=7):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Create sequence data grouped by store
    temporal_features = []
    
    for store in df_input['Store'].unique():
        store_data = df_input[df_input['Store'] == store].sort_values('Date')
        
        if len(store_data) < window_size:
            continue
        
        # Create sequences
        for i in range(window_size, len(store_data)):
            sequence = store_data.iloc[i-window_size:i][['Year', 'Month', 'Day', 'DayOfWeek']].values.astype(np.float32)
            temporal_features.append(sequence)
    
    if not temporal_features:
        print("Warning: No temporal sequences created. Returning zeros.")
        return np.zeros((len(df_input), window_size))
    
    # Convert to tensor
    sequences = torch.FloatTensor(temporal_features).to(device)
    
    # Apply Multi-Head Attention
    mha = MultiHeadAttention(embed_dim=4, num_heads=2).to(device)
    with torch.no_grad():
        attention_output = mha(sequences, sequences, sequences)
    
    # Extract attention weights as temporal parameters
    temporal_params = attention_output.cpu().numpy().mean(axis=1)  # Average across sequence
    
    return temporal_params

print("Temporal parameter extraction function defined.")

Temporal parameter extraction function defined.


In [11]:
print("Extracting temporal parameters for TS-XGBoost...")

# Extract temporal parameters
temporal_params_train = extract_temporal_parameters(df_train_windowed, feature_cols, window_size=7)
print(f"Training temporal parameters shape: {temporal_params_train.shape}")

# Prepare features for TS-XGBoost
window_features = [col for col in df_train_windowed.columns if col.startswith('Sales_lag_')]

# Align temporal parameters with training data
X_train_ts = df_train_windowed[feature_cols + window_features].copy()
y_train_ts = df_train_windowed['Sales'].copy()

# Add temporal parameters if shapes match
if len(temporal_params_train) == len(X_train_ts):
    for i in range(temporal_params_train.shape[1]):
        X_train_ts[f'temporal_param_{i}'] = temporal_params_train[:, i]
    print(f"Temporal parameters added to training features. Final shape: {X_train_ts.shape}")
else:
    print(f"Warning: Temporal params shape {temporal_params_train.shape} doesn't match training data {X_train_ts.shape}")
    print("Proceeding without temporal parameters for TS-XGBoost.")

Extracting temporal parameters for TS-XGBoost...
Training temporal parameters shape: (367690, 4)
Proceeding without temporal parameters for TS-XGBoost.


In [12]:
print("Training TS-XGBoost model...")

# Prepare test data similarly
X_test_ts = df_test_windowed[feature_cols + window_features].copy()
y_test_ts = df_test_windowed['Sales'].copy()

# Train TS-XGBoost
ts_xgb_model = xgb.XGBRegressor(
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)
ts_xgb_model.fit(X_train_ts, y_train_ts, verbose=0)

# Predictions
y_pred_ts_xgb = ts_xgb_model.predict(X_test_ts)

# Calculate RMSPE
rmspe_ts_xgb = calculate_rmspe(y_test_ts, y_pred_ts_xgb)

print(f"TS-XGBoost RMSPE: {rmspe_ts_xgb:.4f}")
print(f"Paper TS-XGBoost RMSPE: 0.253")
print(f"Difference: {abs(rmspe_ts_xgb - 0.253):.4f}")

Training TS-XGBoost model...
TS-XGBoost RMSPE: 0.1432
Paper TS-XGBoost RMSPE: 0.253
Difference: 0.1098


### TS-LGBM

According to paper Section 3.2, TS-LGBM combines:
1. Sliding window synthesis for temporal features.
2. Multi-Head Attention for temporal parameter extraction.
3. LightGBM regression utilizing the extracted temporal parameters.


In [13]:
print("Extracting temporal parameters for TS-LGBM...")

# Extract temporal parameters for training
temporal_params_train_lgb = extract_temporal_parameters(df_train_windowed, feature_cols, window_size=7)
print(f"Training temporal parameters shape: {temporal_params_train_lgb.shape}")

# Prepare features for TS-LGBM
X_train_ts_lgb = df_train_windowed[feature_cols + window_features].copy()
y_train_ts_lgb = df_train_windowed['Sales'].copy()

# Add temporal parameters
if len(temporal_params_train_lgb) == len(X_train_ts_lgb):
    for i in range(temporal_params_train_lgb.shape[1]):
        X_train_ts_lgb[f'temporal_param_lgb_{i}'] = temporal_params_train_lgb[:, i]
    print(f"Temporal parameters added to TS-LGBM training features. Final shape: {X_train_ts_lgb.shape}")
else:
    print(f"Warning: Mismatch in temporal params shape. Using available features.")

Extracting temporal parameters for TS-LGBM...
Training temporal parameters shape: (367690, 4)


In [14]:
print("Training TS-LGBM model...")

# Prepare test data for TS-LGBM
X_test_ts_lgb = df_test_windowed[feature_cols + window_features].copy()
y_test_ts_lgb = df_test_windowed['Sales'].copy()

# Train TS-LGBM
ts_lgb_model = lgb.LGBMRegressor(
    max_depth=-1,
    learning_rate=0.1,
    n_estimators=100,
    boosting_type='gbdt',
    random_state=42,
    n_jobs=-1
)
ts_lgb_model.fit(X_train_ts_lgb, y_train_ts_lgb)

# Predictions
y_pred_ts_lgb = ts_lgb_model.predict(X_test_ts_lgb)

# Calculate RMSPE
rmspe_ts_lgb = calculate_rmspe(y_test_ts_lgb, y_pred_ts_lgb)

print(f"TS-LGBM RMSPE: {rmspe_ts_lgb:.4f}")
print(f"Paper TS-LGBM RMSPE: 0.248")
print(f"Difference: {abs(rmspe_ts_lgb - 0.248):.4f}")

Training TS-LGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010174 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2409
[LightGBM] [Info] Number of data points in the train set: 370595, number of used features: 19
[LightGBM] [Info] Start training from score 5459.465589
TS-LGBM RMSPE: 0.1490
Paper TS-LGBM RMSPE: 0.248
Difference: 0.0990


## Reference Results

Consolidated evaluation results across the four reference architectures.


In [15]:
# Create comparison dataframe
results_comparison = pd.DataFrame([
    {"Model": "XGBoost", "Paper RMSPE": 0.274, **calculate_regression_metrics(y_test, y_pred_xgb)},
    {"Model": "LightGBM", "Paper RMSPE": 0.273, **calculate_regression_metrics(y_test, y_pred_lgb)},
    {"Model": "TS-XGBoost", "Paper RMSPE": 0.253, **calculate_regression_metrics(y_test_ts, y_pred_ts_xgb)},
    {"Model": "TS-LGBM", "Paper RMSPE": 0.248, **calculate_regression_metrics(y_test_ts_lgb, y_pred_ts_lgb)},
])


## Best Baseline & Export (Reference)


In [16]:
display(results_comparison.sort_values("Paper RMSPE"))

,Model,Paper RMSPE,RMSE,MSE,RMSPE,R^2,MAE
3,TS-LGBM,0.248,885.53,784170.32,0.1490,0.9322,581.80
2,TS-XGBoost,0.253,872.63,761485.31,0.1432,0.9342,570.52
1,LightGBM,0.273,1706.33,2911546.91,0.3489,0.7941,1232.47
0,XGBoost,0.274,1801.78,3246403.50,0.3578,0.7704,1287.30


# Exploration Based on Our Preprocessing

This section uses the ACF-based lag selection and rolling mean preprocessing derived from the baseline Pharma datasets to evaluate its impact on Rossmann sales prediction.

## Preprocessing

We calculate the ACF on the aggregated daily sales to find a global `lag_selected`. Then, we apply this lag to generate lag features and a rolling mean per Store.

In [17]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf
import numpy as np
import pandas as pd

print("Computing global max lag based on ACF of aggregate sales...")
df_agg = df_cleaned.groupby("Date")["Sales"].sum().reset_index()
acf_values = acf(df_agg["Sales"], nlags=30)
lag_selected = int(np.argmax(acf_values[1:]) + 1)
print(f"Optimal global lag selected: {lag_selected}")

print("Generating lag and rolling mean features per store...")
df_features = df_cleaned.sort_values(["Store", "Date"]).copy()

for lag in range(1, lag_selected + 1):
    df_features[f"lag_{lag}"] = df_features.groupby("Store")["Sales"].shift(lag)

df_features[f"rolling_mean_{lag_selected}"] = df_features.groupby("Store")["Sales"].shift(1).rolling(window=lag_selected).mean()

df_features.dropna(subset=[f"lag_{lag_selected}", f"rolling_mean_{lag_selected}"], inplace=True)

unique_dates = sorted(df_features["Date"].unique())
train_end_idx = min(900, len(unique_dates) - 42)
train_end_date = unique_dates[train_end_idx]

df_train_new = df_features[df_features["Date"] <= train_end_date].copy()
df_test_new = df_features[df_features["Date"] > train_end_date].copy()

new_feature_cols = feature_cols + [f"lag_{i}" for i in range(1, lag_selected + 1)] + [f"rolling_mean_{lag_selected}"]

X_train_new = df_train_new[new_feature_cols].copy()
y_train_new = df_train_new["Sales"].copy()
X_test_new = df_test_new[new_feature_cols].copy()
y_test_new = df_test_new["Sales"].copy()
print("Data preparation complete.")


Computing global max lag based on ACF of aggregate sales...
Optimal global lag selected: 14
Generating lag and rolling mean features per store...
Data preparation complete.


## Modeling

We retrain XGBoost, LightGBM, TS-XGBoost, and TS-LGBM on the new feature set.

### XGBoost


In [18]:
import xgboost as xgb
import lightgbm as lgb
import torch

print("Training XGBoost on new features...")
xgb_model_new = xgb.XGBRegressor(max_depth=8, learning_rate=0.1, n_estimators=100, random_state=42, n_jobs=-1)
xgb_model_new.fit(X_train_new, y_train_new)
xgb_pred_new = xgb_model_new.predict(X_test_new)
xgb_rmspe_new = calculate_rmspe(y_test_new, xgb_pred_new)
print(f"XGBoost RMSPE (Our Preprocessing): {xgb_rmspe_new:.4f}")


Training XGBoost on new features...
XGBoost RMSPE (Our Preprocessing): 0.1196


### LightGBM


In [19]:
print("Training LightGBM on new features...")
lgb_model_new = lgb.LGBMRegressor(max_depth=-1, learning_rate=0.1, n_estimators=100, boosting_type="gbdt", random_state=42, n_jobs=-1)
lgb_model_new.fit(X_train_new, y_train_new)
lgb_pred_new = lgb_model_new.predict(X_test_new)
lgb_rmspe_new = calculate_rmspe(y_test_new, lgb_pred_new)
print(f"LightGBM RMSPE (Our Preprocessing): {lgb_rmspe_new:.4f}")


Training LightGBM on new features...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.023049 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4461
[LightGBM] [Info] Number of data points in the train set: 828458, number of used features: 28
[LightGBM] [Info] Start training from score 5826.232406
LightGBM RMSPE (Our Preprocessing): 0.1286


### TS-XGBoost


In [20]:
print("Extracting temporal parameters for TS models (Our Preprocessing)...")
temporal_params_train_new = extract_temporal_parameters(df_train_new, feature_cols, window_size=7)
temporal_params_test_new = extract_temporal_parameters(df_test_new, feature_cols, window_size=7)

def drop_first_window(df, window_size=7):
    # Handle pandas deprecation by specifying include_groups=False if using pandas 2.2+, 
    # but standard iloc on groupby works for older versions. To be safe:
    return df.groupby("Store").apply(lambda x: x.iloc[window_size:]).reset_index(drop=True)

df_train_new_windowed = drop_first_window(df_train_new, window_size=7)
df_test_new_windowed = drop_first_window(df_test_new, window_size=7)

X_train_ts_new = df_train_new_windowed[new_feature_cols].copy()
y_train_ts_new = df_train_new_windowed["Sales"].copy()

X_test_ts_new = df_test_new_windowed[new_feature_cols].copy()
y_test_ts_new = df_test_new_windowed["Sales"].copy()

if len(temporal_params_train_new) == len(X_train_ts_new):
    for i in range(temporal_params_train_new.shape[1]):
        X_train_ts_new[f"temporal_param_{i}"] = temporal_params_train_new[:, i]

if len(temporal_params_test_new) == len(X_test_ts_new):
    for i in range(temporal_params_test_new.shape[1]):
        X_test_ts_new[f"temporal_param_{i}"] = temporal_params_test_new[:, i]

print("Training TS-XGBoost on new features...")
ts_xgb_model_new = xgb.XGBRegressor(max_depth=8, learning_rate=0.1, n_estimators=100, random_state=42, n_jobs=-1)
ts_xgb_model_new.fit(X_train_ts_new, y_train_ts_new)
ts_xgb_pred_new = ts_xgb_model_new.predict(X_test_ts_new)
ts_xgb_rmspe_new = calculate_rmspe(y_test_ts_new, ts_xgb_pred_new)
print(f"TS-XGBoost RMSPE (Our Preprocessing): {ts_xgb_rmspe_new:.4f}")


Extracting temporal parameters for TS models (Our Preprocessing)...
Training TS-XGBoost on new features...
TS-XGBoost RMSPE (Our Preprocessing): 0.1629


### TS-LGBM


In [21]:
print("Training TS-LGBM on new features...")
ts_lgb_model_new = lgb.LGBMRegressor(max_depth=-1, learning_rate=0.1, n_estimators=100, random_state=42, n_jobs=-1)
ts_lgb_model_new.fit(X_train_ts_new, y_train_ts_new)
ts_lgb_pred_new = ts_lgb_model_new.predict(X_test_ts_new)
ts_lgb_rmspe_new = calculate_rmspe(y_test_ts_new, ts_lgb_pred_new)
print(f"TS-LGBM RMSPE (Our Preprocessing): {ts_lgb_rmspe_new:.4f}")


Training TS-LGBM on new features...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.057097 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5478
[LightGBM] [Info] Number of data points in the train set: 821920, number of used features: 32
[LightGBM] [Info] Start training from score 5834.250202
TS-LGBM RMSPE (Our Preprocessing): 0.1580


## Best Baseline & Export (Our Preprocessing)


In [22]:
df_compare_new = pd.DataFrame([
    {"Model": "XGBoost", **calculate_regression_metrics(y_test_new, xgb_pred_new)},
    {"Model": "LightGBM", **calculate_regression_metrics(y_test_new, lgb_pred_new)},
    {"Model": "TS-XGBoost", **calculate_regression_metrics(y_test_ts_new, ts_xgb_pred_new)},
    {"Model": "TS-LGBM", **calculate_regression_metrics(y_test_ts_new, ts_lgb_pred_new)},
])

display(df_compare_new.sort_values("RMSPE"))


,Model,RMSE,MSE,RMSPE,R^2,MAE
0,XGBoost,791.44,626370.94,0.1196,0.9562,515.09
1,LightGBM,810.67,657188.64,0.1286,0.9541,535.06
3,TS-LGBM,1006.39,1012829.44,0.1580,0.9334,701.84
2,TS-XGBoost,1021.02,1042491.31,0.1629,0.9315,732.62


## Our Results

Summary of results with the new ACF Lag + Rolling Mean preprocessing:
- XGBoost RMSPE: Calculated above
- LightGBM RMSPE: Calculated above
- TS-XGBoost RMSPE: Calculated above
- TS-LGBM RMSPE: Calculated above


# Summary

## Experimental Setup

Two preprocessing approaches compared using 4 forecasting models (including Attention-based Hybrid models) on the Rossmann daily sales dataset:

| Aspect | Reference (Paper) | Our Preprocessing |
|--------|-------------------|-------------------|
| Split | 900 days / 42 days (train/test) | 900 days / 42 days (train/test) |
| Features | Store, DayOfWeek, Date, Promo, StateHoliday, SchoolHoliday, Year, Month, Day, StoreType, Assortment, CompetitionDistance, Promo2, WeekOfYear | ACF-based lag_n + rolling_mean |
| Temporal | Time-series sliding window (size=7) + Multi-Head Attention | ACF-based lag_n + Multi-Head Attention |
| Hyperparameters | Reference tuned parameters | Reference tuned parameters |

## Models Evaluated

| # | Model | Type |
|---|-------|------|
| 1 | XGBoost | Machine Learning Baseline |
| 2 | LightGBM | Machine Learning Baseline |
| 3 | TS-XGBoost | Attention-based Hybrid |
| 4 | TS-LGBM | Attention-based Hybrid |

## Key Metrics

- RMSPE (Root Mean Square Percentage Error) — primary metric as used in the paper.
- The results are compared between the Reference Preprocessing and Our Preprocessing to evaluate the impact of global lag features on attention-based hybrid models.

## Key Findings

- **Attention Mechanism Utility**: The TS-LGBM and TS-XGBoost architectures demonstrate how Multi-Head Attention can extract complex temporal parameters from raw time-series windows (Year, Month, Day, DayOfWeek) to boost gradient boosting performance.
- **Preprocessing Impact**: Applying the ACF-based lag preprocessing alongside the Attention-based temporal parameter extraction effectively fuses strong, explicit autoregressive lag features with latent deep learning representations.

## Limitations

- **Data Shrinkage**: The sliding window approach natively removes sequential samples (e.g., the first 7 days for every single store) to construct the temporal context, slightly shrinking the effective training period and discarding early observations.
- **Complexity vs Baseline**: Extracting attention-based temporal parameters requires an initialized PyTorch neural network forward pass during training and inference. This heavily increases computational cost, memory overhead, and inference time compared to standalone XGBoost or LightGBM models.
- **Parameter Ambiguity**: Exact attention mechanism hyperparameters (e.g., embedding dimensions, number of heads) were unspecified in the original Zeng et al. paper, necessitating assumptions that may not perfectly align with the authors' proprietary implementation.
